In [ ]:

import glob, subprocess, sys, os, time, json, urllib.request
# install vLLM from the competition wheelhouse (offline)
wh=glob.glob("/kaggle/input/**/arc3-vllm-h100-wheelhouse*", recursive=True) or glob.glob("/kaggle/input/*wheelhouse*")
whl=glob.glob("/kaggle/input/**/requirements.lock", recursive=True)
sp="/kaggle/working/vsp"
if whl:
    subprocess.run([sys.executable,"-m","pip","install","--no-index","--find-links",os.path.dirname(whl[0]),
                    "--requirement",whl[0],"--target",sp,"--upgrade","--no-warn-conflicts"],check=False)
    sys.path.insert(0,sp)
# locate gpt-oss-120b weights
cfg=[os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json",recursive=True) if "gpt-oss" in p.lower() or "gpt_oss" in p.lower()]
MODEL=cfg[0] if cfg else glob.glob("/kaggle/input/*gpt-oss*")[0]
_tok=[os.path.dirname(p) for p in glob.glob("/kaggle/input/**/tokenizer.json",recursive=True)]
TOK=_tok[0] if _tok else MODEL
print("[gptoss] tokenizer dir:", TOK, flush=True)
print("[gptoss] model dir:", MODEL, "| files:", sorted(os.listdir(MODEL))[:8], flush=True)
# serve on the RTX Pro 6000
log=open("/kaggle/working/vllm.log","w")
p=subprocess.Popen([sys.executable,"-m","vllm.entrypoints.openai.api_server","--model",MODEL,
    "--tokenizer",TOK,"--served-model-name","gptoss","--host","127.0.0.1","--port","1234","--enforce-eager",
    "--max-model-len","8192","--gpu-memory-utilization","0.92"],
    env={**os.environ,"PYTHONPATH":sp},stdout=log,stderr=subprocess.STDOUT)
# wait for ready
import subprocess as _s
for i in range(180):
    try:
        m=json.loads(urllib.request.urlopen("http://127.0.0.1:1234/v1/models",timeout=5).read())
        print("[gptoss] SERVED OK:",[x["id"] for x in m.get("data",[])],flush=True); break
    except Exception: time.sleep(10)
else:
    print("[gptoss] SERVE FAILED — tail:",flush=True); print(open("/kaggle/working/vllm.log").read()[-1500:]); 
    import pandas as pd; pd.DataFrame([["1_0","1",True,0]],columns=["row_id","game_id","end_of_game","score"]).to_parquet("/kaggle/working/submission.parquet",index=False); raise SystemExit
# throughput smoke test (a coding-style prompt like the EWM proposer would send)
t0=time.time()
req=urllib.request.Request("http://127.0.0.1:1234/v1/chat/completions",
    data=json.dumps({"model":"gptoss","messages":[{"role":"user","content":"Write a Python function that flood-fills a 2D grid from (r,c) with a new color, 4-connected. Return only code."}],
    "max_tokens":600,"temperature":0.4}).encode(),headers={"Content-Type":"application/json"})
r=json.loads(urllib.request.urlopen(req,timeout=300).read())
out=r["choices"][0]["message"]["content"]; ntok=r.get("usage",{}).get("completion_tokens",0); dt=time.time()-t0
print(f"[gptoss] THROUGHPUT: {ntok} tok in {dt:.1f}s = {ntok/dt:.1f} tok/s",flush=True)
print("[gptoss] sample output (first 300ch):",out[:300],flush=True)
print(f"\n[gptoss] VERDICT: gpt-oss-120b SERVES on RTX Pro 6000 offline; {ntok/dt:.0f} tok/s -> EWM-viable if >~15 tok/s",flush=True)
import pandas as pd; pd.DataFrame([["1_0","1",True,1]],columns=["row_id","game_id","end_of_game","score"]).to_parquet("/kaggle/working/submission.parquet",index=False)
